# 1 import和常量定义

In [ ]:
from pathlib import Path

musicChannelsFilePath = Path(r"C:\02Programmer\02Proj\PyVSCode\ConfigPrivate\Rep001Tools002YTProjV4\musicChannels.txt")

# 2 工具方法

In [ ]:
# 读取musicChannels.txt文件，记录读取到哪一行。
def readFileLine(path: Path, line_number: int = 0, strip_newline: bool = True) -> list[str]:
    """
    读取文本文件中的指定行或所有非空行。

    参数:
        path (Path): 要读取的文件路径。
        line_number (int): 指定要读取的行号（从 1 开始计数）。
                          若为 0 或负数，则返回所有非空行。
        strip_newline (bool): 是否去除行尾的换行符。仅在读取单行时生效。不去的话链接后面就有个\n。

    返回:
        list[str]: 包含所读取行内容的列表。
                   - 如果读取所有非空行，返回多个字符串；
                   - 如果读取指定行，只返回该行构成的列表；
                   - 如果出错或行号越界，返回空列表。

    异常:
        - 文件未找到时会打印提示；
        - 其他读取异常会打印错误信息。
    """
    
    try:
        with path.open('r', encoding='utf-8') as f:
            if line_number <= 0:
                return [line.strip() for line in f if line.strip()]
            else:
                for current_line_num, line in enumerate(f, start=1):
                    if current_line_num == line_number:
                        return [line.rstrip('\n') if strip_newline else line]
    except FileNotFoundError:
        print(f"文件未找到: {path}")
    except Exception as e:
        print(f"读取文件时出错: {e}")
    return []

# readFileLine(musicChannelsFilePath, 1)


In [ ]:
# 输入youtube music播放列表，按要求下载音乐
from pathlib import Path
import subprocess

# 路径设置
cookies_path = Path(r"C:\02Programmer\02Proj\PyVSCode\ConfigPrivate\Rep001Tools002YTProjV4\cookies.txt")     # 用于身份验证的 cookies 路径
output_root = Path(r"G:\YTDowmLoad\musicWithLrc")           # 输出根目录

# yt-dlp 执行器
yt_dlp_binary = "yt-dlp"

# 核心参数组合
use_cookies        = ["--cookies", str(cookies_path)]
extract_audio      = ["--extract-audio"] # 提取音频
audio_format       = ["--audio-format", "mp3"] # 输出为 mp3
audio_quality      = ["--audio-quality", "0"] # 最佳音质（0）
prefer_ffmpeg      = ["--prefer-ffmpeg"] # 使用 ffmpeg
safe_audio_format = ["-f", "ba/best"]  # 使用最佳音频，若失败退回best

# 元数据嵌入相关
embed_metadata     = ["--embed-metadata"] # 写入音频元数据
embed_thumbnail    = ["--embed-thumbnail"] # 插入封面图（缩略图）
add_metadata       = ["--add-metadata"] # 增加上传者/描述等元数据
write_info_json    = ["--write-info-json"] # 保存 info.json 文件以备查验

# 其他参数
replace_artist_commas = ["--replace-in-metadata", "artist", ",", " _"] # 将逗号替换为下划线
view_count_filter  = ["--match-filter", "view_count > 1000000"]  # 仅下载播放量大于100万的视频
download_archive   = ["--download-archive", str(output_root / "musicDownloaded.txt")]  # 已下载记录，避免重复
exclude_filter     = ["--reject-title", "Live"]  # 排除标题中包含"Live"的所有视频
embed_subtitles    = ["--embed-subs"]  # 嵌入字幕
write_subtitles    = ["--write-subs"]  # 下载字幕
lyrics_download    = ["--sub-format", "lrc"]  # 歌词下载参数
no_overwrite       = ["--no-post-overwrites"]  # 不覆盖已存在的文件（防止重复下载）


output_template    = ["--output", str(output_root / "%(artist).40s/%(album)s/%(title)s.%(ext)s")] # 输出路径格式 artist 最多 40 字符
print_filepath     = ["--print", "after_move:filepath"]  # 下载完成后输出音频路径

retry_on_failure   = ["--retries", "5", "--fragment-retries", "5"]  # 自动重试下载失败或片段失败
ignore_errors      = ["--ignore-errors"]  # 遇到错误跳过该视频，不终止整个播放列表，无法解决整个播放列表下完后才报中间部分下载失败的问题，废弃不用
force_key_youtube  = ["--force-key", "youtube"] # 避免链接访问后被重定向，避免不了，废弃不用

utf8_encoding      = ["--encoding", "utf-8"]  # 避免文件名、元数据乱码
concurrent_fragments = ["--concurrent-fragments", "3"]  # 最多同时下载3个文件
show_progress      = ["--progress"]  # 显示下载进度条
verbose_log        = ["--verbose"]  # 输出详细的调试日志

# 命令构造函数
def build_cmd(url: str) -> list[str]:
    return (
        [yt_dlp_binary] +
        use_cookies +
        extract_audio +
        audio_format +
        audio_quality +
        prefer_ffmpeg +
        safe_audio_format +

        embed_metadata +
        embed_thumbnail +
        add_metadata +
        write_info_json +
        
        replace_artist_commas +
        view_count_filter +
        download_archive +
        exclude_filter +
        embed_subtitles +
        write_subtitles +
        lyrics_download +
        no_overwrite +
        
        output_template +
        print_filepath +
        
        retry_on_failure +
        utf8_encoding +
        concurrent_fragments +
        show_progress +
        verbose_log +
        [url]
    )

# 示例使用
example_url = "https://music.youtube.com/playlist?list=OLAK5uy_l1lY8iZBrFbWVTUd9ag6azs7ARlDNCJ2w"
cmd = build_cmd(example_url)

# 然后可以用 subprocess 执行:
# subprocess.run(cmd, check=True)


In [ ]:
def wait_for_user_input():
    """程序暂停，等待用户输入'y'继续或'n'退出"""
    while True:
        user_input = input("输入 'y' 继续，输入 'n' 退出: ")
        if user_input.lower() == 'n':
            return False  # 用户选择退出，返回 False
        elif user_input.lower() == 'y':
            return True  # 用户选择继续，返回 True
        else:
            print("无效输入，请输入 'y' 或 'n'。")
            
# wait_for_user_input()

In [ ]:
# 从 Firefox 获取 cookies 并 格式化为 Netscape 格式（yt-dlp 支持的格式）
import time
import browser_cookie3


def reget_cookies(cookies_path: Path):
    # 从 Firefox 获取 cookies
    cookies = browser_cookie3.firefox(domain_name='youtube.com')

    # 格式化为 Netscape 格式（yt-dlp 支持的格式）
    cookies_txt_path = cookies_path

    with open(cookies_txt_path, 'w', encoding='utf-8') as f:
        f.write("# Netscape HTTP Cookie File\n")
        for cookie in cookies:
            domain = cookie.domain if cookie.domain.startswith('.') else '.' + cookie.domain
            path = cookie.path or '/'
            secure = "TRUE" if cookie.secure else "FALSE"
            expires = int(time.time()) + 3600 * 24 * 30  # 设置过期时间为 30 天
            f.write(f"{domain}\tTRUE\t{path}\t{secure}\t{expires}\t{cookie.name}\t{cookie.value}\n")
    print(f"Cookies 已保存到: {cookies_txt_path}")
    
# reget_cookies(cookies_path)


In [ ]:
# 提取播放列表所有视频链接
def extract_video_urls(playlist_url: str) -> list[str]:
    cmd = [
        "yt-dlp", "--flat-playlist", "--print", "url",
        "--cookies", str(cookies_path),
        playlist_url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    urls = result.stdout.strip().splitlines()
    return [f"https://music.youtube.com/watch?v={u}" if "http" not in u else u for u in urls]

# 示例使用
# extract_video_urls(example_url)

In [ ]:
line_num = 1
already_reget_cookies = False  # 是否需要重新获取 cookies


while True:
    # 读取下一行
    line = readFileLine(musicChannelsFilePath, line_number=line_num)
    if not line:
        print("已经是文件末尾,取不出来链接了。")
        # 如果没有更多行，暂停并等待用户输入，让用户增加更多行
        if not wait_for_user_input():  # 如果用户选择退出，停止程序
            print("没有更多内容，程序终止。")
            break
        # 如果用户输入'y'，继续
        print("继续处理下一行...")
        continue  # 继续处理下一行

    # 处理读取到的行
    playlistUrl = line[0]  # 假设每行只有一个 URL
    extractUrls = extract_video_urls(playlistUrl)
    for url in extractUrls:
        cmd = build_cmd(url)
        try:
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as e:
            print(f"[stderr] {e.stderr}")
            continue
        
    print(f"下载完第 {line_num} 行。")
    line_num += 1  # 更新行号
    
    if 0:
        url = line[0]
        cmd = build_cmd(url)
        try: # 一次下载整个列表，报错会在列表都下完后集中报错
            # 执行命令
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"下载完第 {line_num} 行。")
        except subprocess.CalledProcessError as e:
            print(f"[stderr] {e.stderr}")
            if not already_reget_cookies:
                print(f"下载第 {line_num} 行时 run cmd failed reget cookies ，并重试当前行。")
                reget_cookies(cookies_path)
                already_reget_cookies = True  # 标记为已重试
                continue  # 再次尝试本行
            print(f"下载第 {line_num} 行时，命令执行失败: {e}. 等待用户反馈。")
            if not wait_for_user_input():  # 如果用户选择退出，停止程序
                print("run cmd failed ，程序终止。")
                break
            # 如果用户输入'y'，继续
            print("继续处理下一行...")
            already_reget_cookies = False
            continue
        line_num += 1  # 更新行号